# STRING Cleaning - Chuẩn hóa từ `links` và `allias`/`aliases`

Notebook này chỉ xử lý bộ dữ liệu STRING. Không đọc và không ghi bất kỳ bảng GEO/GDC nào.

Input HDFS:
- `/drugtarget/data/raw/STRING/links/run_date=*/*`
- `/drugtarget/data/raw/STRING/allias/run_date=*/*`, nếu không có thì fallback sang `/drugtarget/data/raw/STRING/aliases/run_date=*/*`

Output HDFS dạng Parquet overwrite:
- `/drugtarget/data/refined/STRING/gene_map`
- `/drugtarget/data/refined/STRING/edges_protein`
- `/drugtarget/data/refined/STRING/edges_gene`
- `/drugtarget/data/refined/STRING/nodes_gene`
- `/drugtarget/data/refined/STRING/nodes_protein`


## 1. Khởi tạo PySpark và kiểm tra worker

Cell này ưu tiên chạy trên Spark Standalone cluster để tận dụng worker. Nếu không phát hiện worker `ALIVE`, notebook in dòng `WARN` và chuyển về `local[*]` để vẫn chạy được trong môi trường đơn máy. Các tham số partition/shuffle được chặn trên dưới để job không tạo quá nhiều task nhỏ hoặc dồn quá nhiều dữ liệu vào một task.


In [ ]:
from datetime import datetime
import glob
import json
import os
import socket
import sys
import time
from urllib import request as urlrequest

# Kernel Jupyter đôi khi chưa có PySpark trong PYTHONPATH.
# Vì Spark đã được cài local, cell này tự thêm Spark/Py4J để notebook chạy trực tiếp được.
SPARK_HOME = os.environ.get('SPARK_HOME', '/home/dis/spark-3.5.1-bin-hadoop3')
os.environ.setdefault('SPARK_HOME', SPARK_HOME)
spark_python_dir = os.path.join(SPARK_HOME, 'python')
if os.path.isdir(spark_python_dir) and spark_python_dir not in sys.path:
    sys.path.insert(0, spark_python_dir)
for py4j_zip in glob.glob(os.path.join(SPARK_HOME, 'python', 'lib', 'py4j-*.zip')):
    if py4j_zip not in sys.path:
        sys.path.insert(0, py4j_zip)

from pyspark import StorageLevel
from pyspark.sql import SparkSession, Window, functions as F, types as T


# Cấu hình Spark cluster hiện tại. Notebook sẽ hỏi Spark Master UI để biết có worker sống hay không.
SPARK_MASTER_HOST = 'master11'
SPARK_MASTER_PORT = 7077
SPARK_CLUSTER_MASTER = f'spark://{SPARK_MASTER_HOST}:{SPARK_MASTER_PORT}'
SPARK_MASTER_WEB_UI = f'http://{SPARK_MASTER_HOST}:8080'
SPARK_DRIVER_HOST = 'master11'

# Profile an toàn cho STRING: links có thể lớn, còn map alias thường nhỏ hơn.
# Giới hạn này giúp chạy song song vừa đủ, tránh nghẽn do shuffle hoặc quá nhiều file output nhỏ.
SAFE_DRIVER_MEMORY = '4g'
SAFE_EXECUTOR_MEMORY = '3g'
SAFE_EXECUTOR_CORES = 1
SAFE_MAX_CORES_PER_WORKER = 2
SAFE_MIN_SHUFFLE_PARTITIONS = 32
SAFE_MAX_SHUFFLE_PARTITIONS = 128
SAFE_MAX_PARTITION_BYTES = '64m'
SAFE_OPEN_COST_BYTES = '8m'
SAFE_SMALL_OUTPUT_PARTITIONS = 8


def _is_spark_master_reachable(host: str, port: int, timeout_seconds: int = 2) -> bool:
    # Kiểm tra cổng Spark master trước khi cố nộp application lên cluster.
    try:
        with socket.create_connection((host, port), timeout=timeout_seconds):
            return True
    except OSError:
        return False


def _get_spark_master_status(timeout_seconds: int = 2) -> dict:
    # Spark Standalone Master UI có endpoint JSON chứa danh sách worker và trạng thái từng worker.
    try:
        with urlrequest.urlopen(f'{SPARK_MASTER_WEB_UI}/json/', timeout=timeout_seconds) as response:
            return json.loads(response.read().decode('utf-8'))
    except Exception as exc:
        print(f'Không đọc được Spark Master UI JSON, sẽ cân nhắc fallback local: {exc}')
        return {}


def _get_alive_workers(master_status: dict) -> list:
    # Chỉ tính worker ALIVE và còn core, vì worker chết hoặc hết core đều không giúp chạy song song.
    workers = master_status.get('workers') or []
    alive_workers = []
    for worker in workers:
        worker_state = str(worker.get('state', 'ALIVE')).upper()
        worker_cores = int(worker.get('cores', worker.get('coresfree', 0)) or 0)
        if worker_state == 'ALIVE' and worker_cores > 0:
            alive_workers.append(worker)
    return alive_workers


def _stop_existing_spark_session() -> None:
    # Khi chạy lại notebook nhiều lần trong cùng kernel, dừng session cũ để tránh reuse nhầm master.
    existing_spark = globals().get('spark')
    if existing_spark is None:
        return
    try:
        existing_spark.stop()
        time.sleep(2)
    except Exception as exc:
        print(f'Không thể stop SparkSession cũ, tiếp tục tạo session mới: {exc}')


def _get_executor_count(spark_session) -> int:
    # getExecutorMemoryStatus có cả driver, nên trừ 1 để ra số executor thực tế.
    try:
        memory_status_size = spark_session.sparkContext._jsc.sc().getExecutorMemoryStatus().size()
        return max(memory_status_size - 1, 0)
    except Exception:
        return 0


def _wait_for_executors(spark_session, timeout_seconds: int = 20) -> int:
    # Spark có thể mất vài giây để cấp executor sau khi app bắt đầu, nên đợi ngắn trước khi fallback.
    deadline = time.time() + timeout_seconds
    executor_count = _get_executor_count(spark_session)
    while time.time() < deadline and executor_count == 0:
        time.sleep(2)
        executor_count = _get_executor_count(spark_session)
    return executor_count


def _safe_shuffle_partitions(alive_worker_count: int) -> int:
    # Mỗi worker nhận khoảng 24 shuffle partition, nhưng luôn nằm trong ngưỡng an toàn đã cấu hình.
    if alive_worker_count <= 0:
        return SAFE_MIN_SHUFFLE_PARTITIONS
    return min(max(alive_worker_count * 24, SAFE_MIN_SHUFFLE_PARTITIONS), SAFE_MAX_SHUFFLE_PARTITIONS)


def _build_spark_session(master_url: str, shuffle_partitions: int, use_cluster: bool):
    # Cấu hình I/O và adaptive execution giúp Spark tự coalesce partition sau shuffle khi phù hợp.
    builder = (
        SparkSession.builder.appName('drugtarget-string-cleaning')
        .master(master_url)
        .config('spark.sql.session.timeZone', 'UTC')
        .config('spark.sql.parquet.compression.codec', 'snappy')
        .config('spark.sql.adaptive.enabled', 'true')
        .config('spark.sql.adaptive.coalescePartitions.enabled', 'true')
        .config('spark.sql.files.maxPartitionBytes', SAFE_MAX_PARTITION_BYTES)
        .config('spark.sql.files.openCostInBytes', SAFE_OPEN_COST_BYTES)
        .config('spark.sql.shuffle.partitions', str(shuffle_partitions))
        .config('spark.default.parallelism', str(shuffle_partitions))
        .config('spark.hadoop.fs.defaultFS', 'hdfs://master11:9000')
        .config('spark.driver.memory', SAFE_DRIVER_MEMORY)
        .config('spark.pyspark.python', 'python3')
        .config('spark.executorEnv.PYSPARK_PYTHON', 'python3')
    )
    if use_cluster:
        builder = (
            builder.config('spark.driver.host', SPARK_DRIVER_HOST)
            .config('spark.executor.memory', SAFE_EXECUTOR_MEMORY)
            .config('spark.executor.cores', str(SAFE_EXECUTOR_CORES))
            .config('spark.cores.max', str(max(SAFE_MAX_CORES_PER_WORKER, shuffle_partitions // 12)))
        )
    spark_session = builder.getOrCreate()
    spark_session.sparkContext.setLogLevel('WARN')
    return spark_session


# Kiểm tra worker trước khi tạo SparkSession. Không có worker thì WARN và chạy local[*].
_stop_existing_spark_session()
master_status = _get_spark_master_status()
alive_workers = _get_alive_workers(master_status)
alive_worker_count = len(alive_workers)
master_reachable = _is_spark_master_reachable(SPARK_MASTER_HOST, SPARK_MASTER_PORT)
use_cluster = master_reachable and alive_worker_count > 0

if not use_cluster:
    print('WARN: Không phát hiện Spark worker ALIVE, chuyển sang local[*].')

shuffle_partitions = _safe_shuffle_partitions(alive_worker_count if use_cluster else 0)
spark = _build_spark_session(
    SPARK_CLUSTER_MASTER if use_cluster else 'local[*]',
    shuffle_partitions,
    use_cluster,
)

# Nếu master có worker nhưng app không nhận executor, dừng app cluster và chuyển local để tránh treo job.
if use_cluster:
    executor_count = _wait_for_executors(spark, timeout_seconds=20)
    if executor_count == 0:
        print('WARN: Spark master có worker nhưng app không nhận executor, chuyển sang local[*].')
        spark.stop()
        time.sleep(2)
        use_cluster = False
        shuffle_partitions = _safe_shuffle_partitions(0)
        spark = _build_spark_session('local[*]', shuffle_partitions, use_cluster=False)

SPARK_MODE = 'CLUSTER' if use_cluster else 'LOCAL_FALLBACK'
SPARK_EXECUTOR_COUNT = _get_executor_count(spark)
SAFE_EDGE_OUTPUT_PARTITIONS = min(max(shuffle_partitions, SAFE_MIN_SHUFFLE_PARTITIONS), SAFE_MAX_SHUFFLE_PARTITIONS)

print(f'Spark mode: {SPARK_MODE}')
print(f'Spark master đang dùng: {spark.sparkContext.master}')
print(f'Số worker ALIVE phát hiện từ master UI: {alive_worker_count}')
print(f'Số executor hiện có, không tính driver: {SPARK_EXECUTOR_COUNT}')
print(f'spark.sql.shuffle.partitions = {spark.conf.get("spark.sql.shuffle.partitions")}')
print(f'SAFE_EDGE_OUTPUT_PARTITIONS = {SAFE_EDGE_OUTPUT_PARTITIONS}')


## 2. Khai báo HDFS path và chọn input alias

Dữ liệu raw có thể dùng tên thư mục `allias` theo yêu cầu hiện tại hoặc `aliases` theo chính tả thường gặp trong manifest STRING. Cell này kiểm tra wildcard trên HDFS, ưu tiên `allias`, rồi fallback sang `aliases` nếu cần.


In [ ]:
# Khai báo HDFS URI đầy đủ để notebook không phụ thuộc defaultFS của shell hay kernel.
HDFS_BASE_URI = 'hdfs://master11:9000'
RAW_STRING_ROOT = f'{HDFS_BASE_URI}/drugtarget/data/raw/STRING'
REFINED_STRING_ROOT = f'{HDFS_BASE_URI}/drugtarget/data/refined/STRING'

LINKS_INPUT_PATH = f'{RAW_STRING_ROOT}/links/run_date=*/*'
ALIAS_ALLIAS_INPUT_PATH = f'{RAW_STRING_ROOT}/allias/run_date=*/*'
ALIAS_ALIASES_INPUT_PATH = f'{RAW_STRING_ROOT}/aliases/run_date=*/*'

OUTPUT_PATHS = {
    'gene_map': f'{REFINED_STRING_ROOT}/gene_map',
    'edges_protein': f'{REFINED_STRING_ROOT}/edges_protein',
    'edges_gene': f'{REFINED_STRING_ROOT}/edges_gene',
    'nodes_gene': f'{REFINED_STRING_ROOT}/nodes_gene',
    'nodes_protein': f'{REFINED_STRING_ROOT}/nodes_protein',
}

# Ép Hadoop/Spark dùng đúng NameNode HDFS; nếu không, path /drugtarget có thể bị hiểu thành local path.
hadoop_conf = spark.sparkContext._jsc.hadoopConfiguration()
hadoop_conf.set('fs.defaultFS', HDFS_BASE_URI)
fs = spark._jvm.org.apache.hadoop.fs.FileSystem.get(hadoop_conf)
Path = spark._jvm.org.apache.hadoop.fs.Path


def hdfs_glob_has_files(pattern: str) -> bool:
    # Dùng globStatus để kiểm tra wildcard run_date=* trước khi Spark đọc dữ liệu.
    try:
        statuses = fs.globStatus(Path(pattern))
        if statuses is None:
            return False
        return any(not status.isDirectory() for status in statuses)
    except Exception as exc:
        print(f'Không kiểm tra được HDFS pattern {pattern}: {exc}')
        return False


def pick_alias_input_path() -> str:
    # Ưu tiên allias theo yêu cầu; nếu HDFS dùng aliases thì tự động chuyển sang aliases.
    if hdfs_glob_has_files(ALIAS_ALLIAS_INPUT_PATH):
        print(f'Dùng input alias theo path allias: {ALIAS_ALLIAS_INPUT_PATH}')
        return ALIAS_ALLIAS_INPUT_PATH
    if hdfs_glob_has_files(ALIAS_ALIASES_INPUT_PATH):
        print(f'Không thấy allias, fallback sang aliases: {ALIAS_ALIASES_INPUT_PATH}')
        return ALIAS_ALIASES_INPUT_PATH
    raise FileNotFoundError(
        'Không tìm thấy input alias ở cả allias và aliases: '
        f'{ALIAS_ALLIAS_INPUT_PATH} | {ALIAS_ALIASES_INPUT_PATH}'
    )


if not hdfs_glob_has_files(LINKS_INPUT_PATH):
    raise FileNotFoundError(f'Không tìm thấy input links: {LINKS_INPUT_PATH}')

ALIAS_INPUT_PATH = pick_alias_input_path()

print(f'LINKS_INPUT_PATH : {LINKS_INPUT_PATH}')
print(f'ALIAS_INPUT_PATH : {ALIAS_INPUT_PATH}')
for table_name, table_path in OUTPUT_PATHS.items():
    print(f'Output {table_name}: {table_path}')


## 3. Đọc raw `links` và `allias`/`aliases`

`links` thường là file có khoảng trắng giữa `protein1 protein2 combined_score`. `aliases` thường dùng tab giữa `protein_id alias source`. Cell này đọc bằng `text` để chủ động bỏ header/comment và parse an toàn, kể cả khi file đã được nén hoặc gom trong nhiều `run_date`.


In [ ]:
def normalize_gene_name(col):
    # Chuẩn hóa gene symbol để join/order ổn định: trim, bỏ khoảng trắng dư và viết hoa.
    return F.upper(F.regexp_replace(F.trim(col.cast('string')), r'\s+', ''))


def extract_ensp_id(col):
    # STRING protein_id thường có dạng 9606.ENSP..., nên cần tách ENSP để dùng ở output.
    extracted = F.regexp_extract(col.cast('string'), r'(ENSP[0-9]+(?:\.[0-9]+)?)', 1)
    return F.when(extracted == '', F.lit(None).cast('string')).otherwise(extracted)


# Đọc links bằng text để parse theo whitespace. Header protein1/protein2/combined_score bị bỏ sau parse.
links_lines = (
    spark.read.text(LINKS_INPUT_PATH)
    .select(F.trim(F.col('value')).alias('line'))
    .filter(F.col('line').isNotNull() & (F.col('line') != '') & (~F.col('line').startswith('#')))
)

links_raw = (
    links_lines
    .withColumn('parts', F.split(F.col('line'), r'\s+'))
    .filter(F.size(F.col('parts')) >= 3)
    .select(
        F.element_at(F.col('parts'), 1).alias('protein_id_src'),
        F.element_at(F.col('parts'), 2).alias('protein_id_dst'),
        F.element_at(F.col('parts'), 3).alias('combined_score_raw'),
    )
    .filter(~F.lower(F.col('protein_id_src')).isin('protein1', 'protein_id_src', '#protein1'))
    .withColumn('combined_score_protein', F.col('combined_score_raw').cast('double'))
    .filter(F.col('protein_id_src').isNotNull() & F.col('protein_id_dst').isNotNull())
    .filter(F.col('combined_score_protein').isNotNull())
    .drop('combined_score_raw')
    .persist(StorageLevel.DISK_ONLY)
)

# Đọc alias bằng text. Nếu dòng có tab thì tách bằng tab, nếu không thì fallback whitespace.
# Chỉ dùng cột source để lấy Ensembl_gene và Ensembl_HGNC_symbol theo yêu cầu.
alias_lines = (
    spark.read.text(ALIAS_INPUT_PATH)
    .select(F.trim(F.col('value')).alias('line'))
    .filter(F.col('line').isNotNull() & (F.col('line') != '') & (~F.col('line').startswith('#')))
)

aliases_base = (
    alias_lines
    .withColumn('tab_parts', F.split(F.col('line'), '\t'))
    .withColumn('space_parts', F.split(F.col('line'), r'\s+'))
    .withColumn('parts', F.when(F.size(F.col('tab_parts')) >= 3, F.col('tab_parts')).otherwise(F.col('space_parts')))
    .filter(F.size(F.col('parts')) >= 3)
    .select(
        F.element_at(F.col('parts'), 1).alias('protein_id'),
        F.element_at(F.col('parts'), 2).alias('alias_value'),
        F.element_at(F.col('parts'), 3).alias('source'),
    )
    .filter(~F.lower(F.col('protein_id')).isin('string_protein_id', 'protein_id', '#string_protein_id'))
    .filter(F.col('protein_id').isNotNull() & F.col('alias_value').isNotNull() & F.col('source').isNotNull())
    .persist(StorageLevel.DISK_ONLY)
)

# Chỉ giữ 2 source alias phục vụ map gene theo yêu cầu: ENSG và HGNC symbol.
# Các source khác như PDB, UniProt, Entrez, description... không tham gia gene_map.
ALIASES_KEEP_SOURCES = ['ensembl_gene', 'ensembl_hgnc_symbol']
aliases_gene_sources = (
    aliases_base
    .withColumn('source_norm', F.lower(F.trim(F.col('source'))))
    .filter(F.col('source_norm').isin(ALIASES_KEEP_SOURCES))
    .select('protein_id', 'alias_value', 'source', 'source_norm')
    .persist(StorageLevel.DISK_ONLY)
)

print(f'Raw links parsed rows       : {links_raw.count()}')
print(f'Raw aliases parsed rows     : {aliases_base.count()}')
print(f'Alias rows dùng cho gene_map: {aliases_gene_sources.count()}')
aliases_gene_sources.groupBy('source').count().orderBy('source').show(truncate=False)


## 4. Tạo `gene_map`

`gene_id` lấy từ alias có `source = Ensembl_gene`; `gene_name` lấy từ alias có `source = Ensembl_HGNC_symbol`. Mỗi `protein_id` được gom về một dòng duy nhất, nên bảng này là bản đồ protein-gene chuẩn để join cho các bước sau.


In [ ]:
# Lấy danh sách protein xuất hiện trong 2 source alias liên quan để không kéo các alias PDB/UniProt/description vào gene_map.
# Protein thiếu gene_id vẫn được giữ nếu có HGNC symbol; protein thiếu gene_name vẫn được giữ nếu có ENSG.
proteins_from_alias = (
    aliases_gene_sources
    .select('protein_id')
    .distinct()
    .withColumn('ensp_id', extract_ensp_id(F.col('protein_id')))
)

# ENSG từ source Ensembl_gene. Nếu một protein có nhiều alias cùng source, chọn min để kết quả deterministic.
gene_id_candidates = (
    aliases_gene_sources
    .filter(F.col('source_norm') == 'ensembl_gene')
    .filter(F.col('alias_value').rlike('^ENSG'))
    .groupBy('protein_id')
    .agg(F.min(F.col('alias_value')).alias('gene_id'))
)

# HGNC symbol từ source Ensembl_HGNC_symbol. Chọn min để tránh duplicate protein_id ở output.
gene_name_candidates = (
    aliases_gene_sources
    .filter(F.col('source_norm') == 'ensembl_hgnc_symbol')
    .groupBy('protein_id')
    .agg(F.min(F.col('alias_value')).alias('gene_name'))
)

gene_map = (
    proteins_from_alias
    .join(gene_id_candidates, on='protein_id', how='left')
    .join(gene_name_candidates, on='protein_id', how='left')
    .withColumn('gene_name_norm', normalize_gene_name(F.col('gene_name')))
    .withColumn(
        'gene_confidence',
        F.when(F.col('gene_id').isNotNull() & F.col('gene_name').isNotNull(), F.lit('high'))
        .when(F.col('gene_id').isNotNull() | F.col('gene_name').isNotNull(), F.lit('medium'))
        .otherwise(F.lit('low')),
    )
    .select('protein_id', 'ensp_id', 'gene_id', 'gene_name', 'gene_name_norm', 'gene_confidence')
    .persist(StorageLevel.DISK_ONLY)
)

print(f'gene_map rows: {gene_map.count()}')
gene_map.groupBy('gene_confidence').count().orderBy('gene_confidence').show(truncate=False)
gene_map.show(5, truncate=False)


## 5. Tạo `edges_protein`

`edges_protein` đi trực tiếp từ raw `links`. `combined_score` của STRING nằm trong thang 0-1000, nên `edge_weight_protein` được chuẩn hóa bằng `combined_score / 1000.0` để tiện dùng làm trọng số đồ thị.


In [ ]:
edges_protein = (
    links_raw
    .withColumn('ensp_id_src', extract_ensp_id(F.col('protein_id_src')))
    .withColumn('ensp_id_dst', extract_ensp_id(F.col('protein_id_dst')))
    .withColumn('edge_weight_protein', (F.col('combined_score_protein').cast('double') / F.lit(1000.0)))
    .select(
        'protein_id_src',
        'protein_id_dst',
        'ensp_id_src',
        'ensp_id_dst',
        'combined_score_protein',
        'edge_weight_protein',
    )
    .persist(StorageLevel.DISK_ONLY)
)

print(f'edges_protein rows: {edges_protein.count()}')
edges_protein.show(5, truncate=False)


## 6. Tạo `edges_gene`

Mỗi cạnh protein được left join với `gene_map` ở cả hai đầu để tạo intermediate `edges_protein_gene_joined` và không làm mất cạnh protein khi một protein thiếu `gene_id`. Sau đó `edges_gene` chỉ lấy các cạnh có đủ `gene_id_src` và `gene_id_dst`, canonical hóa cặp theo `gene_id`, rồi group thành một cạnh gene-level cho mỗi cặp ENSG.


In [ ]:
# Chuẩn bị hai bản gene_map với tên cột riêng cho đầu src và dst của cạnh protein.
gene_map_src = gene_map.select(
    F.col('protein_id').alias('protein_id_src'),
    F.col('gene_id').alias('gene_id_src'),
    F.col('gene_name').alias('gene_name_src'),
    F.col('gene_name_norm').alias('gene_name_norm_src'),
)
gene_map_dst = gene_map.select(
    F.col('protein_id').alias('protein_id_dst'),
    F.col('gene_id').alias('gene_id_dst'),
    F.col('gene_name').alias('gene_name_dst'),
    F.col('gene_name_norm').alias('gene_name_norm_dst'),
)

# Left join để giữ nguyên toàn bộ cạnh protein. Các dòng thiếu gene_id vẫn nằm ở protein-level,
# nhưng sẽ bị lọc khỏi gene-level graph ở bước edges_gene phía dưới.
edges_protein_gene_joined = (
    edges_protein
    .join(gene_map_src, on='protein_id_src', how='left')
    .join(gene_map_dst, on='protein_id_dst', how='left')
    .select(
        'protein_id_src',
        'protein_id_dst',
        'ensp_id_src',
        'ensp_id_dst',
        'gene_id_src',
        'gene_id_dst',
        'gene_name_src',
        'gene_name_dst',
        'gene_name_norm_src',
        'gene_name_norm_dst',
        'combined_score_protein',
        'edge_weight_protein',
    )
    .persist(StorageLevel.DISK_ONLY)
)

# edges_gene chỉ dùng các cạnh có đủ ENSG hai phía. Cặp gene được canonical hóa bằng gene_id
# để downstream GDC join sạch với gene_id_base = gene_id.
gene_edges_for_group = (
    edges_protein_gene_joined
    .filter(F.col('gene_id_src').isNotNull() & F.col('gene_id_dst').isNotNull())
    .filter(F.col('gene_id_src') != F.col('gene_id_dst'))
)

edge_gene_ordered = (
    gene_edges_for_group
    .withColumn('src_first_by_gene_id', F.col('gene_id_src') <= F.col('gene_id_dst'))
    .select(
        F.when(F.col('src_first_by_gene_id'), F.col('gene_id_src')).otherwise(F.col('gene_id_dst')).alias('gene_id_src'),
        F.when(F.col('src_first_by_gene_id'), F.col('gene_id_dst')).otherwise(F.col('gene_id_src')).alias('gene_id_dst'),
        F.when(F.col('src_first_by_gene_id'), F.col('gene_name_src')).otherwise(F.col('gene_name_dst')).alias('gene_name_src'),
        F.when(F.col('src_first_by_gene_id'), F.col('gene_name_dst')).otherwise(F.col('gene_name_src')).alias('gene_name_dst'),
        F.when(F.col('src_first_by_gene_id'), F.col('gene_name_norm_src')).otherwise(F.col('gene_name_norm_dst')).alias('gene_name_norm_src'),
        F.when(F.col('src_first_by_gene_id'), F.col('gene_name_norm_dst')).otherwise(F.col('gene_name_norm_src')).alias('gene_name_norm_dst'),
        F.when(F.col('src_first_by_gene_id'), F.col('ensp_id_src')).otherwise(F.col('ensp_id_dst')).alias('edge_ensp_id_src'),
        F.when(F.col('src_first_by_gene_id'), F.col('ensp_id_dst')).otherwise(F.col('ensp_id_src')).alias('edge_ensp_id_dst'),
        F.col('combined_score_protein'),
        F.col('edge_weight_protein'),
    )
)

gene_pair_cols = ['gene_id_src', 'gene_id_dst']
pair_window = Window.partitionBy(*gene_pair_cols)
rank_window = pair_window.orderBy(
    F.desc('combined_score_protein'),
    F.asc('edge_ensp_id_src'),
    F.asc('edge_ensp_id_dst'),
)

# Window vừa tính aggregate cho gene pair, vừa chọn protein edge tốt nhất để lấy best ENSP và gene_name.
edges_gene_internal = (
    edge_gene_ordered
    .withColumn('max_combined_score_gene', F.max('combined_score_protein').over(pair_window))
    .withColumn('max_edge_weight_gene', F.max('edge_weight_protein').over(pair_window))
    .withColumn('protein_edge_count', F.count(F.lit(1)).over(pair_window).cast('long'))
    .withColumn('_rn', F.row_number().over(rank_window))
    .filter(F.col('_rn') == 1)
    .select(
        'gene_name_src',
        'gene_name_dst',
        'gene_name_norm_src',
        'gene_name_norm_dst',
        'gene_id_src',
        'gene_id_dst',
        F.col('edge_ensp_id_src').alias('best_ensp_id_src'),
        F.col('edge_ensp_id_dst').alias('best_ensp_id_dst'),
        F.col('max_combined_score_gene').cast('double').alias('max_combined_score_gene'),
        F.col('max_edge_weight_gene').cast('double').alias('max_edge_weight_gene'),
        'protein_edge_count',
    )
    .persist(StorageLevel.DISK_ONLY)
)

edges_gene = (
    edges_gene_internal
    .select(
        'gene_name_src',
        'gene_name_dst',
        'gene_id_src',
        'gene_id_dst',
        'best_ensp_id_src',
        'best_ensp_id_dst',
        'max_combined_score_gene',
        'max_edge_weight_gene',
        'protein_edge_count',
    )
    .persist(StorageLevel.DISK_ONLY)
)

print(f'edges_protein_gene_joined rows: {edges_protein_gene_joined.count()}')
print(f'edges_gene rows               : {edges_gene.count()}')
edges_protein_gene_joined.show(5, truncate=False)
edges_gene.show(5, truncate=False)


## 7. Tạo `nodes_gene` và `nodes_protein`

`nodes_gene` tính degree từ `edges_gene` theo `gene_id`, rồi join rollup từ `gene_map` để bổ sung `protein_count` và `protein_ids`. `nodes_protein` vẫn lấy union toàn bộ protein từ `gene_map` và hai đầu `edges_protein`, nên protein thiếu `gene_id` không bị mất ở protein-level output.


In [ ]:
# Degree gene: dùng gene_id từ edges_gene_internal để khớp với graph gene-level canonical theo ENSG.
gene_incident_edges = (
    edges_gene_internal
    .select(
        F.col('gene_id_src').alias('gene_id'),
        F.col('gene_id_dst').alias('neighbor_gene_id'),
        F.col('max_edge_weight_gene').alias('edge_weight'),
    )
    .unionByName(
        edges_gene_internal.select(
            F.col('gene_id_dst').alias('gene_id'),
            F.col('gene_id_src').alias('neighbor_gene_id'),
            F.col('max_edge_weight_gene').alias('edge_weight'),
        )
    )
)

gene_degree = (
    gene_incident_edges
    .groupBy('gene_id')
    .agg(
        F.countDistinct('neighbor_gene_id').cast('long').alias('degree_gene'),
        F.sum('edge_weight').cast('double').alias('weighted_degree_gene'),
    )
)

# Rollup gene-protein từ gene_map theo gene_id. Protein thiếu gene_id vẫn giữ ở protein-level,
# nhưng không tạo node gene vì không thể join sạch với GDC gene_id_base.
gene_protein_rollup = (
    gene_map
    .filter(F.col('gene_id').isNotNull())
    .groupBy('gene_id')
    .agg(
        F.min('gene_name').alias('gene_name'),
        F.min('gene_name_norm').alias('gene_name_norm'),
        F.countDistinct('protein_id').cast('long').alias('protein_count'),
        F.sort_array(F.collect_set('protein_id')).alias('protein_ids'),
    )
)

nodes_gene = (
    gene_protein_rollup
    .join(gene_degree, on='gene_id', how='left')
    .select(
        'gene_name',
        'gene_name_norm',
        'gene_id',
        F.coalesce(F.col('degree_gene'), F.lit(0).cast('long')).alias('degree_gene'),
        F.coalesce(F.col('weighted_degree_gene'), F.lit(0.0)).alias('weighted_degree_gene'),
        'protein_count',
        'protein_ids',
    )
    .persist(StorageLevel.DISK_ONLY)
)

# Degree protein: dùng cạnh protein vô hướng, tính distinct neighbor để tránh đếm trùng neighbor.
protein_incident_edges = (
    edges_protein
    .select(
        F.col('protein_id_src').alias('protein_id'),
        F.col('protein_id_dst').alias('neighbor_protein_id'),
        F.col('edge_weight_protein').alias('edge_weight'),
    )
    .unionByName(
        edges_protein.select(
            F.col('protein_id_dst').alias('protein_id'),
            F.col('protein_id_src').alias('neighbor_protein_id'),
            F.col('edge_weight_protein').alias('edge_weight'),
        )
    )
)

protein_degree = (
    protein_incident_edges
    .groupBy('protein_id')
    .agg(
        F.countDistinct('neighbor_protein_id').cast('long').alias('degree_protein'),
        F.sum('edge_weight').cast('double').alias('weighted_degree_protein'),
    )
)

# nodes_protein gồm protein có trong gene_map hoặc links; protein_name null vì không đọc file details.
all_protein_nodes = (
    gene_map.select('protein_id')
    .unionByName(edges_protein.select(F.col('protein_id_src').alias('protein_id')))
    .unionByName(edges_protein.select(F.col('protein_id_dst').alias('protein_id')))
    .distinct()
    .withColumn('ensp_id_from_protein_id', extract_ensp_id(F.col('protein_id')))
)

gene_map_for_nodes = gene_map.select(
    'protein_id',
    F.col('ensp_id').alias('ensp_id_from_gene_map'),
    'gene_id',
    'gene_name',
    'gene_name_norm',
)

nodes_protein = (
    all_protein_nodes
    .join(gene_map_for_nodes, on='protein_id', how='left')
    .join(protein_degree, on='protein_id', how='left')
    .select(
        'protein_id',
        F.coalesce(F.col('ensp_id_from_gene_map'), F.col('ensp_id_from_protein_id')).alias('ensp_id'),
        F.lit(None).cast('string').alias('protein_name'),
        'gene_id',
        'gene_name',
        'gene_name_norm',
        F.coalesce(F.col('degree_protein'), F.lit(0).cast('long')).alias('degree_protein'),
        F.coalesce(F.col('weighted_degree_protein'), F.lit(0.0)).alias('weighted_degree_protein'),
    )
    .persist(StorageLevel.DISK_ONLY)
)

print(f'nodes_gene rows   : {nodes_gene.count()}')
print(f'nodes_protein rows: {nodes_protein.count()}')
nodes_gene.show(5, truncate=False)
nodes_protein.show(5, truncate=False)


## 8. Kiểm tra chất lượng trước khi ghi

Các assert fail sớm nếu dữ liệu lệch schema/logic: `gene_map` không được trùng `protein_id`; score/weight protein phải nằm trong khoảng hợp lệ; `edges_gene` không có self-loop và phải theo thứ tự alphabet; degree/weighted degree không âm.


In [ ]:
def assert_zero(value: int, message: str) -> None:
    # Helper assert có in số lỗi cụ thể để debug nhanh khi dữ liệu raw thay đổi.
    if value != 0:
        raise AssertionError(f'{message}. Số bản ghi lỗi: {value}')


def assert_columns(df, expected_columns: list, table_name: str) -> None:
    # Kiểm tra thứ tự cột trước khi ghi để downstream đọc Parquet ổn định.
    actual_columns = df.columns
    if actual_columns != expected_columns:
        raise AssertionError(f'{table_name} sai schema. Expected={expected_columns}; Actual={actual_columns}')


def assert_double_column(df, column_name: str, table_name: str) -> None:
    # Các cột điểm tương tác và trọng số phải là double để phục vụ graph/ML downstream.
    data_type = df.schema[column_name].dataType
    if not isinstance(data_type, T.DoubleType):
        raise AssertionError(f'{table_name}.{column_name} phải là double, hiện tại là {data_type.simpleString()}')


expected_columns = {
    'gene_map': ['protein_id', 'ensp_id', 'gene_id', 'gene_name', 'gene_name_norm', 'gene_confidence'],
    'edges_protein': ['protein_id_src', 'protein_id_dst', 'ensp_id_src', 'ensp_id_dst', 'combined_score_protein', 'edge_weight_protein'],
    'edges_gene': ['gene_name_src', 'gene_name_dst', 'gene_id_src', 'gene_id_dst', 'best_ensp_id_src', 'best_ensp_id_dst', 'max_combined_score_gene', 'max_edge_weight_gene', 'protein_edge_count'],
    'nodes_gene': ['gene_name', 'gene_name_norm', 'gene_id', 'degree_gene', 'weighted_degree_gene', 'protein_count', 'protein_ids'],
    'nodes_protein': ['protein_id', 'ensp_id', 'protein_name', 'gene_id', 'gene_name', 'gene_name_norm', 'degree_protein', 'weighted_degree_protein'],
}

expected_intermediate_columns = [
    'protein_id_src',
    'protein_id_dst',
    'ensp_id_src',
    'ensp_id_dst',
    'gene_id_src',
    'gene_id_dst',
    'gene_name_src',
    'gene_name_dst',
    'gene_name_norm_src',
    'gene_name_norm_dst',
    'combined_score_protein',
    'edge_weight_protein',
]

tables = {
    'gene_map': gene_map.select(*expected_columns['gene_map']),
    'edges_protein': edges_protein.select(*expected_columns['edges_protein']),
    'edges_gene': edges_gene.select(*expected_columns['edges_gene']),
    'nodes_gene': nodes_gene.select(*expected_columns['nodes_gene']),
    'nodes_protein': nodes_protein.select(*expected_columns['nodes_protein']),
}

for table_name, df in tables.items():
    assert_columns(df, expected_columns[table_name], table_name)
assert_columns(edges_protein_gene_joined, expected_intermediate_columns, 'edges_protein_gene_joined')

assert_double_column(tables['edges_protein'], 'combined_score_protein', 'edges_protein')
assert_double_column(tables['edges_protein'], 'edge_weight_protein', 'edges_protein')
assert_double_column(tables['edges_gene'], 'max_combined_score_gene', 'edges_gene')
assert_double_column(tables['edges_gene'], 'max_edge_weight_gene', 'edges_gene')
assert_double_column(edges_protein_gene_joined, 'combined_score_protein', 'edges_protein_gene_joined')
assert_double_column(edges_protein_gene_joined, 'edge_weight_protein', 'edges_protein_gene_joined')

duplicate_protein_ids = (
    gene_map
    .groupBy('protein_id')
    .count()
    .filter(F.col('count') > 1)
    .count()
)
assert_zero(duplicate_protein_ids, 'gene_map có duplicate protein_id')

edges_protein_count = edges_protein.count()
edges_protein_gene_joined_count = edges_protein_gene_joined.count()
if edges_protein_gene_joined_count != edges_protein_count:
    raise AssertionError(
        'edges_protein_gene_joined phải giữ nguyên số dòng edges_protein sau left join. '
        f'edges_protein={edges_protein_count}; joined={edges_protein_gene_joined_count}'
    )

missing_nodes_protein = (
    all_protein_nodes
    .select('protein_id')
    .join(nodes_protein.select('protein_id'), on='protein_id', how='left_anti')
    .count()
)
assert_zero(missing_nodes_protein, 'nodes_protein thiếu protein từ union gene_map và edges_protein')

invalid_protein_scores = (
    edges_protein
    .filter(
        (~F.col('combined_score_protein').between(0, 1000))
        | (~F.col('edge_weight_protein').between(0.0, 1.0))
    )
    .count()
)
assert_zero(invalid_protein_scores, 'edges_protein có score/weight ngoài khoảng hợp lệ')

invalid_gene_edges = (
    edges_gene
    .filter(
        F.col('gene_id_src').isNull()
        | F.col('gene_id_dst').isNull()
        | (F.col('gene_id_src') == F.col('gene_id_dst'))
        | (F.col('gene_id_src') > F.col('gene_id_dst'))
    )
    .count()
)
assert_zero(invalid_gene_edges, 'edges_gene có gene_id null, self-loop hoặc gene_id_src > gene_id_dst')

invalid_gene_degrees = (
    nodes_gene
    .filter(F.col('gene_id').isNull() | (F.col('degree_gene') < 0) | (F.col('weighted_degree_gene') < 0))
    .count()
)
assert_zero(invalid_gene_degrees, 'nodes_gene có gene_id null, degree hoặc weighted_degree âm')

invalid_protein_degrees = (
    nodes_protein
    .filter((F.col('degree_protein') < 0) | (F.col('weighted_degree_protein') < 0))
    .count()
)
assert_zero(invalid_protein_degrees, 'nodes_protein có degree hoặc weighted_degree âm')

print('PASS: Tất cả kiểm tra chất lượng STRING đều đạt.')


## 9. Ghi Parquet overwrite và đọc lại kiểm tra

Output được ghi Parquet Snappy theo đúng 5 bảng refined STRING. Bảng cạnh có thể lớn nên được repartition ở mức an toàn; bảng node/map nhỏ hơn thì coalesce để hạn chế quá nhiều file nhỏ trên HDFS.


In [ ]:
def prepare_for_write(table_name: str, df):
    # Cạnh thường lớn hơn node/map, nên chia đều theo partition an toàn trước khi ghi.
    if table_name in {'edges_protein', 'edges_gene'}:
        return df.repartition(SAFE_EDGE_OUTPUT_PARTITIONS)
    return df.coalesce(SAFE_SMALL_OUTPUT_PARTITIONS)


def write_parquet_overwrite(table_name: str, df, path: str) -> None:
    # Ghi overwrite rõ ràng để mỗi lần chạy notebook tạo lại snapshot refined nhất quán.
    prepared_df = prepare_for_write(table_name, df)
    prepared_df.write.mode('overwrite').option('compression', 'snappy').parquet(path)
    print(f'Đã ghi overwrite bảng {table_name}: {path}')


write_start = datetime.utcnow()
print(f'Bắt đầu ghi output STRING lúc UTC: {write_start.isoformat()}Z')

for table_name, df in tables.items():
    write_parquet_overwrite(table_name, df, OUTPUT_PATHS[table_name])

write_end = datetime.utcnow()
print(f'Hoàn tất ghi output STRING lúc UTC: {write_end.isoformat()}Z')
print(f'Thời gian ghi: {(write_end - write_start).total_seconds():.2f} giây')

# Đọc lại đủ 5 bảng để xác nhận Parquet tồn tại, schema đúng và lấy row count thực tế sau ghi.
for table_name, table_path in OUTPUT_PATHS.items():
    print('=' * 100)
    print(f'Bảng: {table_name}')
    print(f'Đường dẫn: {table_path}')
    reloaded_df = spark.read.parquet(table_path)
    reloaded_df.printSchema()
    print(f'Row count: {reloaded_df.count()}')


## 10. Dọn tài nguyên Spark

Cell cuối dừng Spark app để trả tài nguyên cho worker hoặc local runtime sau khi đã ghi và kiểm tra đủ output.


In [ ]:
# Dừng Spark app sau khi hoàn tất để giải phóng tài nguyên cluster/local.
for cached_df_name in ['links_raw', 'aliases_base', 'aliases_gene_sources', 'gene_map', 'edges_protein', 'edges_protein_gene_joined', 'edges_gene_internal', 'edges_gene', 'nodes_gene', 'nodes_protein']:
    cached_df = globals().get(cached_df_name)
    if cached_df is not None:
        try:
            cached_df.unpersist()
        except Exception:
            pass

try:
    spark.stop()
    print('Đã stop Spark app STRING')
except Exception as exc:
    print(f'Không thể stop Spark app STRING: {exc}')
